## 과제 C: 가상 날씨 API (실무 빈도 ★★★★☆)

```python
class WeatherInfo(BaseModel):
    city: str
    temperature_c: float
    condition: Literal["맑음", "흐림", "비", "눈"]
    humidity_percent: int

@tool
def get_weather(city: str) -> WeatherInfo:
    """도시의 현재 날씨를 조회합니다."""
    ...
```

테스트: "서울이랑 부산 날씨 알려줘"   ← 도구를 두 번 호출!

> 💡 _ 심화 노트북(`Day2_Lesson4_API_Practice.ipynb`)에서 **진짜 날씨 API**로 만드는 법을 배웁니다.

In [1]:
# 1️⃣ 실행 환경 감지
try:
    import google.colab
    IS_COLAB = True
    print("✅ Google Colab")
except ImportError:
    IS_COLAB = False
    print("✅ 로컬 Jupyter")

# 3️⃣ API 키 로드
import os

if IS_COLAB:
    from google.colab import userdata
    os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")
else:
    from dotenv import load_dotenv
    load_dotenv()

print("✅ API 키 OK" if os.environ.get("OPENAI_API_KEY") else "❌ 키 없음")

from typing import Literal, Optional
from pydantic import BaseModel, Field   # 1-2교시 정형화 도구
from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage, AIMessage, ToolMessage
from langchain_core.tools import tool   # ⭐ 오늘의 주인공

# Tool Calling은 temperature=0 권장 (도구 선택의 일관성)
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

print("✅ import 완료")

✅ 로컬 Jupyter
✅ API 키 OK
✅ import 완료


In [2]:
def run_with_tools(question: str, tools: list, llm_obj=llm, max_steps: int = 5):
    """도구 호출이 끝날 때까지 반복 실행하고 최종 답변 반환"""
    llm_with_tools = llm_obj.bind_tools(tools)
    tool_map = {t.name: t for t in tools}
    messages = [HumanMessage(content=question)]

    for _ in range(max_steps):
        response = llm_with_tools.invoke(messages)
        messages.append(response)

        if not response.tool_calls:          
            return response.content

        for tc in response.tool_calls:
            result = tool_map[tc["name"]].invoke(tc["args"])

            error = getattr(result, "error", None)
            if error:
                return str(error)

            messages.append(
                ToolMessage(content=str(result), tool_call_id=tc["id"])
            )

    return "최대 단계 수를 초과했습니다."

print("✅ 헬퍼 함수 준비 완료")

✅ 헬퍼 함수 준비 완료


In [3]:
class WeatherInfo(BaseModel):
    city: str
    temperature_c: float
    condition: Literal["맑음", "흐림", "비", "눈"]
    humidity_percent: int
    error: Optional[str] = None

@tool
def get_weather(city: str) -> WeatherInfo:
    """도시의 현재 날씨를 조회합니다."""

    # 실제 API 호출 대신 예시 데이터를 반환합니다.
    example_data = {
        "서울": WeatherInfo(city="서울", temperature_c=22.5, condition="맑음", humidity_percent=55),
        "부산": WeatherInfo(city="부산", temperature_c=18.0, condition="흐림", humidity_percent=70),
    }

    if city in example_data:
        return example_data[city]
    else:
        return WeatherInfo(
            city=city,
            temperature_c=0.0,
            condition="맑음",
            humidity_percent=0,
            error=f"'{city}'의 날씨 정보를 찾을 수 없습니다."
        )

In [4]:
q = "서울이랑 부산 날씨 알려줘"

answer = run_with_tools(q, [get_weather])
print(f"질문: {q}\n답변: {answer}")

질문: 서울이랑 부산 날씨 알려줘
답변: 현재 서울의 날씨는 다음과 같습니다:
- 온도: 22.5°C
- 날씨: 맑음
- 습도: 55%

부산의 날씨는 다음과 같습니다:
- 온도: 18.0°C
- 날씨: 흐림
- 습도: 70%
